# TrustQueryNet Colab Runner

This notebook is the canonical Colab workflow for the TrustQueryNet HAM10000 experiments. It provisions a fresh runtime, prepares the dataset, builds Colab-specific configs, and exposes the main experiment tracks used for baselines, Q1 checks, final evidence, and export packaging.

Run the setup and dataset sections first. The heavy training and export cells use explicit boolean toggles so the notebook stays safe to reopen and easy to reuse.


In [ ]:
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

REPO_URL = "https://github.com/stelioszach03/TrustQueryNet.git"
USE_DRIVE = True
RESET_PROJECT_ROOT = True
DOWNLOAD_HAM10000 = False
DEFAULT_NUM_WORKERS = 0

FULL_SEEDS = [42, 52, 62, 72, 82]
SHORT_SEEDS = [42, 52, 62]
SMOKE_SEEDS = [42]

PROJECT_ROOT = Path("/content/TrustQueryNet")
DRIVE_MOUNT_POINT = Path("/content/drive")
DRIVE_ROOT = DRIVE_MOUNT_POINT / "MyDrive"

HAM_DIR = DRIVE_ROOT / "HAM10000" if USE_DRIVE else Path("/content/HAM10000")
WORKSPACE_ROOT = DRIVE_ROOT / "TrustQueryNet" if USE_DRIVE else Path("/content/TrustQueryNetRuntime")
ARTIFACTS_ROOT = WORKSPACE_ROOT / "artifacts"
ABLATIONS_ROOT = WORKSPACE_ROOT / "ablations"
TABLES_ROOT = WORKSPACE_ROOT / "paper_tables"
EXPORTS_ROOT = WORKSPACE_ROOT / "exports"
ISIC_ROOT = DRIVE_ROOT / "ISIC2019_external_test" if USE_DRIVE else Path("/content/ISIC2019_external_test")

print(f"Project root:   {PROJECT_ROOT}")
print(f"HAM10000 root:  {HAM_DIR}")
print(f"Artifacts root: {ARTIFACTS_ROOT}")
print(f"Ablations root: {ABLATIONS_ROOT}")
print(f"Tables root:    {TABLES_ROOT}")
print(f"Exports root:   {EXPORTS_ROOT}")
print(f"ISIC root:      {ISIC_ROOT}")


## Runtime Setup

Mount Google Drive when you want persisted datasets and artifacts. Set `USE_DRIVE = False` in the configuration cell to keep everything under `/content` for an ephemeral Colab session.


In [ ]:
if USE_DRIVE:
    from google.colab import drive

    drive.mount(str(DRIVE_MOUNT_POINT))
    print(f"Drive mounted at {DRIVE_MOUNT_POINT}")
else:
    print("Skipping Drive mount; using only ephemeral /content paths.")


In [ ]:
import yaml

def ensure_workspace_dirs():
    for path in [WORKSPACE_ROOT, ARTIFACTS_ROOT, ABLATIONS_ROOT, TABLES_ROOT, EXPORTS_ROOT]:
        path.mkdir(parents=True, exist_ok=True)

def run(command, *, cwd=None):
    command = [str(part) for part in command]
    print("$", " ".join(command))
    subprocess.check_call(command, cwd=str(cwd) if cwd else None)

def run_py(script_name, *args, cwd=PROJECT_ROOT):
    run([sys.executable, PROJECT_ROOT / "scripts" / script_name, *args], cwd=cwd)

def sync_repo(reset=RESET_PROJECT_ROOT):
    os.chdir("/content")
    if reset and PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)
    if not PROJECT_ROOT.exists():
        run(["git", "clone", REPO_URL, PROJECT_ROOT], cwd="/content")
    else:
        run(["git", "pull", "--ff-only"], cwd=PROJECT_ROOT)
    run([sys.executable, "-m", "pip", "install", "-q", "-r", PROJECT_ROOT / "requirements-colab.txt"])
    run([sys.executable, "-m", "pip", "install", "-q", "-e", PROJECT_ROOT, "--no-deps"])
    os.chdir(PROJECT_ROOT)
    print(f"Project root ready: {PROJECT_ROOT}")

def guarded_run(enabled, label, callback, *args, **kwargs):
    if enabled:
        print(f"Running: {label}")
        return callback(*args, **kwargs)
    print(f"Skipped: {label}")
    return None

def set_nested(cfg, dotted_key, value):
    target = cfg
    parts = dotted_key.split(".")
    for part in parts[:-1]:
        target = target[part]
    target[parts[-1]] = value

def write_colab_config(base_name, *, output_root, out_name=None, num_workers=DEFAULT_NUM_WORKERS, overrides=None):
    base_path = PROJECT_ROOT / "configs" / base_name
    cfg = yaml.safe_load(base_path.read_text())
    cfg["dataset"]["metadata_csv"] = str(HAM_DIR / "HAM10000_metadata.csv")
    cfg["dataset"]["image_dir"] = str(HAM_DIR / "images")
    cfg["dataset"]["split_csv"] = str(HAM_DIR / "splits.csv")
    cfg["dataset"]["save_split_csv"] = str(HAM_DIR / "splits.csv")
    cfg["output_dir"] = str(output_root)
    cfg["num_workers"] = num_workers
    for key, value in (overrides or {}).items():
        set_nested(cfg, key, value)
    config_name = out_name or f"colab_{Path(base_name).stem}.yaml"
    out_path = PROJECT_ROOT / "configs" / config_name
    out_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    return out_path, cfg

def prepare_ham10000():
    if DOWNLOAD_HAM10000:
        run_py("download_ham10000.py", "--output-root", HAM_DIR)
    metadata_csv = HAM_DIR / "HAM10000_metadata.csv"
    image_dir = HAM_DIR / "images"
    split_csv = HAM_DIR / "splits.csv"
    report_path = ARTIFACTS_ROOT / "ham10000_dataset_report.json"
    if not metadata_csv.exists():
        raise FileNotFoundError(f"Missing metadata CSV: {metadata_csv}")
    if not image_dir.exists():
        raise FileNotFoundError(f"Missing image directory: {image_dir}")
    run_py(
        "prepare_ham10000.py",
        "--metadata-csv",
        metadata_csv,
        "--image-dir",
        image_dir,
        "--split-csv",
        split_csv,
        "--report-json",
        report_path,
    )
    jpg_count = sum(1 for _ in image_dir.glob("*.jpg"))
    print(f"Metadata CSV: {metadata_csv}")
    print(f"Image dir:    {image_dir}")
    print(f"Split CSV:    {split_csv}")
    print(f"JPG count:    {jpg_count}")
    print(f"Report path:  {report_path}")
    return report_path

def run_experiment(config_path):
    run([sys.executable, PROJECT_ROOT / "scripts" / "run_experiment.py", "--config", config_path], cwd=PROJECT_ROOT)

def run_multiseed(config_path, seeds, *, resume_existing=True):
    command = [
        sys.executable,
        PROJECT_ROOT / "scripts" / "run_multiseed_experiment.py",
        "--config",
        config_path,
        "--seeds",
        *[str(seed) for seed in seeds],
    ]
    if resume_existing:
        command.append("--resume-existing")
    run(command, cwd=PROJECT_ROOT)

def ensure_isic2019_download():
    required = [
        ISIC_ROOT / "ISIC_2019_Test_GroundTruth.csv",
        ISIC_ROOT / "ISIC_2019_Test_Metadata.csv",
        ISIC_ROOT / "images",
    ]
    if all(path.exists() for path in required):
        print(f"ISIC2019 external test set already available under {ISIC_ROOT}")
        return
    run_py("download_isic2019_external.py", "--output-root", ISIC_ROOT)

def run_external_validation(multiseed_run_dir, *, num_workers=DEFAULT_NUM_WORKERS):
    run_py(
        "run_external_validation.py",
        "--multiseed-run-dir",
        multiseed_run_dir,
        "--ground-truth-csv",
        ISIC_ROOT / "ISIC_2019_Test_GroundTruth.csv",
        "--metadata-csv",
        ISIC_ROOT / "ISIC_2019_Test_Metadata.csv",
        "--image-dir",
        ISIC_ROOT / "images",
        "--num-workers",
        num_workers,
    )

def show_text(path):
    path = Path(path)
    if not path.exists():
        print(f"Missing: {path}")
        return
    print(f"\n===== {path} =====\n")
    print(path.read_text())

def zip_directory(source_dir, zip_path):
    source_dir = Path(source_dir)
    zip_path = Path(zip_path)
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for file_path in sorted(source_dir.rglob("*")):
            if file_path.is_file():
                archive.write(file_path, arcname=str(file_path.relative_to(source_dir.parent)))
    print(f"Created zip: {zip_path}")
    return zip_path

def package_figure_bundle():
    figure_dir = ARTIFACTS_ROOT / "paper_figures"
    zip_path = ARTIFACTS_ROOT / "paper_figures_bundle.zip"
    required = [
        "internal_reliability_calibrated.png",
        "internal_risk_coverage_calibrated.png",
        "external_reliability_calibrated.png",
        "external_risk_coverage_calibrated.png",
        "internal_external_summary.png",
        "figure_manifest.json",
        "overlap_report.json",
    ]
    missing = [name for name in required if not (figure_dir / name).exists()]
    if missing:
        raise FileNotFoundError(f"Missing figure files: {missing}")
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for name in required:
            archive.write(figure_dir / name, arcname=f"paper_figures/{name}")
    print(f"Created: {zip_path}")
    return zip_path

def package_final_bundle():
    bundle_dir = EXPORTS_ROOT / "trustquerynet-final-hardening-bundle"
    run_names = [
        "q1-ham10000-convnext-repair-final-e12-multiseed",
        "q1-ham10000-convnext-no-repair-final-e12-multiseed",
        "q1-ham10000-convnext-random-repair-final-e12-multiseed",
        "q1-ham10000-convnext-clean-upper-final-e12-multiseed",
        "q1-ham10000-convnext-gce-no-repair-final-e12-multiseed",
        "q1-ham10000-convnext-repair-final-e12-multiseed-external-isic2019-test",
        "q1-ham10000-convnext-no-repair-final-e12-multiseed-external-isic2019-test",
        "q1-ham10000-convnext-random-repair-final-e12-multiseed-external-isic2019-test",
    ]
    keep_run_files = [
        "aggregate_results.json",
        "aggregate_results.csv",
        "aggregate_results.md",
        "seed_results.csv",
        "seed_results.md",
        "multiseed_manifest.json",
        "resolved_base_config.yaml",
    ]
    if bundle_dir.exists():
        shutil.rmtree(bundle_dir)
    bundle_dir.mkdir(parents=True, exist_ok=True)

    runs_out = bundle_dir / "final_evidence"
    runs_out.mkdir(parents=True, exist_ok=True)
    for run_name in run_names:
        src_dir = ARTIFACTS_ROOT / "final_evidence" / run_name
        dst_dir = runs_out / run_name
        dst_dir.mkdir(parents=True, exist_ok=True)
        if not src_dir.exists():
            print(f"Missing run directory: {src_dir}")
            continue
        for rel_name in keep_run_files:
            src_file = src_dir / rel_name
            if src_file.exists():
                shutil.copy2(src_file, dst_dir / rel_name)

    overlap_out = bundle_dir / "overlap"
    overlap_out.mkdir(parents=True, exist_ok=True)
    overlap_root = ARTIFACTS_ROOT / "overlap" / "ham10000-isic2019"
    for name in ["overlap_report.json", "exact_matches.csv", "near_duplicate_candidates.csv"]:
        src_file = overlap_root / name
        if src_file.exists():
            shutil.copy2(src_file, overlap_out / name)

    tables_out = bundle_dir / "paper_tables"
    tables_out.mkdir(parents=True, exist_ok=True)
    for table_dir_name in ["internal_main", "noisy_anchor", "external_main"]:
        src_dir = TABLES_ROOT / table_dir_name
        dst_dir = tables_out / table_dir_name
        if src_dir.exists():
            shutil.copytree(src_dir, dst_dir, dirs_exist_ok=True)

    figures_out = bundle_dir / "paper_figures"
    figures_out.mkdir(parents=True, exist_ok=True)
    for name in ["figure_manifest.json", "overlap_report.json"]:
        src_file = ARTIFACTS_ROOT / "paper_figures" / name
        if src_file.exists():
            shutil.copy2(src_file, figures_out / name)

    zip_path = zip_directory(bundle_dir, EXPORTS_ROOT / "trustquerynet-final-hardening-bundle.zip")
    return bundle_dir, zip_path


In [ ]:
ensure_workspace_dirs()
sync_repo()


## Dataset Preparation

Run this section once per runtime after mounting storage. It optionally downloads HAM10000 and always regenerates the split CSV plus dataset report used by the experiment configs.


In [ ]:
prepare_report_path = prepare_ham10000()
print(f"Prepared dataset report: {prepare_report_path}")


## Baseline Runs And Internal Ablations

These configs cover the pilot run, the full balanced baseline, and the two baseline ablations used to test the value of weighting and active querying.


In [ ]:
pilot_cfg, _ = write_colab_config(
    "pilot_ham10000.yaml",
    output_root=ARTIFACTS_ROOT / "runs" / "pilot-ham10000",
    out_name="colab_pilot_ham10000.yaml",
)
full_cfg, _ = write_colab_config(
    "full_ham10000_convnext.yaml",
    output_root=ARTIFACTS_ROOT / "runs" / "full-ham10000-convnext-balanced",
    out_name="colab_full_ham10000_convnext.yaml",
)
no_weighted_cfg, _ = write_colab_config(
    "full_ham10000_convnext_no_weighted.yaml",
    output_root=ARTIFACTS_ROOT / "runs" / "full-ham10000-convnext-no-weighted",
    out_name="colab_full_ham10000_convnext_no_weighted.yaml",
)
no_querying_cfg, _ = write_colab_config(
    "full_ham10000_convnext_no_querying.yaml",
    output_root=ARTIFACTS_ROOT / "runs" / "full-ham10000-convnext-no-querying",
    out_name="colab_full_ham10000_convnext_no_querying.yaml",
)

for label, path in [
    ("Pilot config", pilot_cfg),
    ("Full config", full_cfg),
    ("No weighted config", no_weighted_cfg),
    ("No querying config", no_querying_cfg),
]:
    print(f"{label}: {path}")


In [ ]:
baseline_jobs = [
    {"enabled": False, "label": "Pilot single run", "mode": "single", "config": pilot_cfg},
    {"enabled": False, "label": "Full single run", "mode": "single", "config": full_cfg},
    {"enabled": False, "label": "Full balanced multiseed", "mode": "multiseed", "config": full_cfg, "seeds": FULL_SEEDS},
    {"enabled": False, "label": "No weighted sampler multiseed", "mode": "multiseed", "config": no_weighted_cfg, "seeds": FULL_SEEDS},
    {"enabled": False, "label": "No querying multiseed", "mode": "multiseed", "config": no_querying_cfg, "seeds": FULL_SEEDS},
]

for job in baseline_jobs:
    if job["mode"] == "single":
        guarded_run(job["enabled"], job["label"], run_experiment, job["config"])
    else:
        guarded_run(job["enabled"], job["label"], run_multiseed, job["config"], job["seeds"])


In [ ]:
EXPORT_BASELINE_ABLATION = False

guarded_run(
    EXPORT_BASELINE_ABLATION,
    "Baseline ablation table export",
    run_py,
    "export_ablation_table.py",
    "--runs-root",
    ARTIFACTS_ROOT / "runs",
    "--run-spec",
    "full-ham10000-convnext-balanced-multiseed::Full model",
    "--run-spec",
    "full-ham10000-convnext-no-weighted-multiseed::No weighted sampler",
    "--run-spec",
    "full-ham10000-convnext-no-querying-multiseed::No query rounds",
    "--output-dir",
    ABLATIONS_ROOT / "ham10000",
)

for path in [
    ARTIFACTS_ROOT / "runs" / "full-ham10000-convnext-balanced-multiseed" / "aggregate_results.md",
    ARTIFACTS_ROOT / "runs" / "full-ham10000-convnext-no-weighted-multiseed" / "aggregate_results.md",
    ARTIFACTS_ROOT / "runs" / "full-ham10000-convnext-no-querying-multiseed" / "aggregate_results.md",
    ABLATIONS_ROOT / "ham10000" / "ablation_table.md",
]:
    show_text(path)


## Q1 Phase-Gate Smoke Checks

This section builds small smoke-test configs for the repair, no-repair, and random-repair variants, then optionally runs the external check and phase-gate verifier.


In [ ]:
smoke_repair_cfg, _ = write_colab_config(
    "q1_ham10000_convnext_repair.yaml",
    output_root=ARTIFACTS_ROOT / "smoke_runs" / "q1-ham10000-convnext-repair-smoke",
    out_name="colab_smoke_q1_ham10000_convnext_repair.yaml",
    overrides={
        "training.epochs": 2,
        "training.early_stopping_patience": 1,
    },
)
smoke_no_repair_cfg, _ = write_colab_config(
    "q1_ham10000_convnext_no_repair.yaml",
    output_root=ARTIFACTS_ROOT / "smoke_runs" / "q1-ham10000-convnext-no-repair-smoke",
    out_name="colab_smoke_q1_ham10000_convnext_no_repair.yaml",
    overrides={
        "training.epochs": 2,
        "training.early_stopping_patience": 1,
    },
)
smoke_random_repair_cfg, _ = write_colab_config(
    "q1_ham10000_convnext_random_repair.yaml",
    output_root=ARTIFACTS_ROOT / "smoke_runs" / "q1-ham10000-convnext-random-repair-smoke",
    out_name="colab_smoke_q1_ham10000_convnext_random_repair.yaml",
    overrides={
        "training.epochs": 2,
        "training.early_stopping_patience": 1,
    },
)

for label, path in [
    ("Repair smoke config", smoke_repair_cfg),
    ("No repair smoke config", smoke_no_repair_cfg),
    ("Random repair smoke config", smoke_random_repair_cfg),
]:
    print(f"{label}: {path}")


In [ ]:
smoke_jobs = [
    {"enabled": False, "label": "Repair smoke multiseed", "config": smoke_repair_cfg, "seeds": SMOKE_SEEDS},
    {"enabled": False, "label": "No repair smoke multiseed", "config": smoke_no_repair_cfg, "seeds": SMOKE_SEEDS},
    {"enabled": False, "label": "Random repair smoke multiseed", "config": smoke_random_repair_cfg, "seeds": SMOKE_SEEDS},
]

for job in smoke_jobs:
    guarded_run(job["enabled"], job["label"], run_multiseed, job["config"], job["seeds"])


In [ ]:
DOWNLOAD_ISIC2019 = False
RUN_SMOKE_EXTERNAL_VALIDATION = False
RUN_SMOKE_PHASE_GATE = False

guarded_run(DOWNLOAD_ISIC2019, "Download ISIC2019 external test set", ensure_isic2019_download)
guarded_run(
    RUN_SMOKE_EXTERNAL_VALIDATION,
    "Smoke external validation (repair)",
    run_external_validation,
    ARTIFACTS_ROOT / "smoke_runs" / "q1-ham10000-convnext-repair-smoke-multiseed",
)
guarded_run(
    RUN_SMOKE_PHASE_GATE,
    "Q1 phase-gate verification",
    run_py,
    "verify_q1_phase_gate.py",
    "--repair-config",
    smoke_repair_cfg,
    "--no-repair-config",
    smoke_no_repair_cfg,
    "--random-repair-config",
    smoke_random_repair_cfg,
    "--repair-run-dir",
    ARTIFACTS_ROOT / "smoke_runs" / "q1-ham10000-convnext-repair-smoke-multiseed" / "seed-42",
    "--random-repair-run-dir",
    ARTIFACTS_ROOT / "smoke_runs" / "q1-ham10000-convnext-random-repair-smoke-multiseed" / "seed-42",
    "--external-run-dir",
    ARTIFACTS_ROOT / "smoke_runs" / "q1-ham10000-convnext-repair-smoke-multiseed-external-isic2019-test",
    "--expected-checkpoint-policy",
    "best_val_macro_f1",
)

smoke_report_path = ARTIFACTS_ROOT / "smoke_runs" / "q1-ham10000-convnext-repair-smoke-multiseed" / "seed-42" / "active_learning_report.json"
if smoke_report_path.exists():
    report = json.loads(smoke_report_path.read_text())
    payload = {
        "repair_protocol": report["final_metrics"].get("repair_protocol"),
        "selected_checkpoint": report["final_metrics"].get("selected_checkpoint"),
        "test_calibrated": {
            key: report["final_metrics"]["test_calibrated"].get(key)
            for key in [
                "accuracy",
                "macro_f1",
                "ece",
                "macro_auroc",
                "aurc",
                "coverage_at_0.5",
                "risk_at_0.5",
            ]
        },
    }
    print(json.dumps(payload, indent=2))
else:
    print(f"Smoke report not found yet: {smoke_report_path}")


## Recipe Lock

Use these runs to compare the repair recipe at 12 and 20 epochs under the same Colab-managed dataset and artifact layout.


In [ ]:
recipe_e12_cfg, _ = write_colab_config(
    "q1_ham10000_convnext_repair.yaml",
    output_root=ARTIFACTS_ROOT / "recipe_lock" / "q1-ham10000-convnext-repair-e12",
    out_name="colab_q1-ham10000-convnext-repair-e12.yaml",
    overrides={
        "training.epochs": 12,
        "training.early_stopping_patience": 4,
    },
)
recipe_e20_cfg, _ = write_colab_config(
    "q1_ham10000_convnext_repair.yaml",
    output_root=ARTIFACTS_ROOT / "recipe_lock" / "q1-ham10000-convnext-repair-e20",
    out_name="colab_q1-ham10000-convnext-repair-e20.yaml",
    overrides={
        "training.epochs": 20,
        "training.early_stopping_patience": 4,
    },
)

print(f"Recipe e12 config: {recipe_e12_cfg}")
print(f"Recipe e20 config: {recipe_e20_cfg}")


In [ ]:
recipe_jobs = [
    {"enabled": False, "label": "Repair recipe lock e12", "config": recipe_e12_cfg, "seeds": SHORT_SEEDS},
    {"enabled": False, "label": "Repair recipe lock e20", "config": recipe_e20_cfg, "seeds": SHORT_SEEDS},
]

for job in recipe_jobs:
    guarded_run(job["enabled"], job["label"], run_multiseed, job["config"], job["seeds"])

for path in [
    ARTIFACTS_ROOT / "recipe_lock" / "q1-ham10000-convnext-repair-e12-multiseed" / "aggregate_results.md",
    ARTIFACTS_ROOT / "recipe_lock" / "q1-ham10000-convnext-repair-e20-multiseed" / "aggregate_results.md",
]:
    show_text(path)


## Final Evidence

These configs produce the final internal and external evidence bundle used for the main TrustQueryNet comparison across repair variants and noisy-label anchors.


In [ ]:
final_cfgs = {}
final_specs = [
    ("q1_ham10000_convnext_repair.yaml", "q1-ham10000-convnext-repair-final-e12"),
    ("q1_ham10000_convnext_no_repair.yaml", "q1-ham10000-convnext-no-repair-final-e12"),
    ("q1_ham10000_convnext_random_repair.yaml", "q1-ham10000-convnext-random-repair-final-e12"),
    ("q1_ham10000_convnext_clean_upper.yaml", "q1-ham10000-convnext-clean-upper-final-e12"),
    ("q1_ham10000_convnext_gce_no_repair.yaml", "q1-ham10000-convnext-gce-no-repair-final-e12"),
]

for base_name, run_name in final_specs:
    config_path, _ = write_colab_config(
        base_name,
        output_root=ARTIFACTS_ROOT / "final_evidence" / run_name,
        out_name=f"colab_{run_name}.yaml",
        overrides={
            "training.epochs": 12,
            "training.early_stopping_patience": 4,
        },
    )
    final_cfgs[run_name] = config_path
    print(f"{run_name}: {config_path}")


In [ ]:
final_jobs = [
    {"enabled": False, "label": "Repair final multiseed", "run_name": "q1-ham10000-convnext-repair-final-e12", "seeds": FULL_SEEDS},
    {"enabled": False, "label": "No repair final multiseed", "run_name": "q1-ham10000-convnext-no-repair-final-e12", "seeds": FULL_SEEDS},
    {"enabled": False, "label": "Random repair final multiseed", "run_name": "q1-ham10000-convnext-random-repair-final-e12", "seeds": FULL_SEEDS},
    {"enabled": False, "label": "Clean upper final multiseed", "run_name": "q1-ham10000-convnext-clean-upper-final-e12", "seeds": SHORT_SEEDS},
    {"enabled": False, "label": "GCE no repair final multiseed", "run_name": "q1-ham10000-convnext-gce-no-repair-final-e12", "seeds": SHORT_SEEDS},
]

for job in final_jobs:
    guarded_run(job["enabled"], job["label"], run_multiseed, final_cfgs[job["run_name"]], job["seeds"])


In [ ]:
DOWNLOAD_ISIC2019 = False
final_external_jobs = [
    {
        "enabled": False,
        "label": "Repair external validation",
        "run_dir": ARTIFACTS_ROOT / "final_evidence" / "q1-ham10000-convnext-repair-final-e12-multiseed",
    },
    {
        "enabled": False,
        "label": "No repair external validation",
        "run_dir": ARTIFACTS_ROOT / "final_evidence" / "q1-ham10000-convnext-no-repair-final-e12-multiseed",
    },
    {
        "enabled": False,
        "label": "Random repair external validation",
        "run_dir": ARTIFACTS_ROOT / "final_evidence" / "q1-ham10000-convnext-random-repair-final-e12-multiseed",
    },
]

guarded_run(DOWNLOAD_ISIC2019, "Download ISIC2019 external test set", ensure_isic2019_download)
for job in final_external_jobs:
    guarded_run(job["enabled"], job["label"], run_external_validation, job["run_dir"])


In [ ]:
RUN_OVERLAP_AUDIT = False
EXPORT_PAPER_TABLES = False
EXPORT_RESULTS_BUNDLE = False
EXPORT_PAPER_FIGURES = False

guarded_run(
    RUN_OVERLAP_AUDIT,
    "HAM10000 / ISIC2019 overlap audit",
    run_py,
    "audit_ham10000_isic2019_overlap.py",
    "--ham-metadata-csv",
    HAM_DIR / "HAM10000_metadata.csv",
    "--ham-image-dir",
    HAM_DIR / "images",
    "--ham-split-csv",
    HAM_DIR / "splits.csv",
    "--isic-ground-truth-csv",
    ISIC_ROOT / "ISIC_2019_Test_GroundTruth.csv",
    "--isic-image-dir",
    ISIC_ROOT / "images",
    "--isic-metadata-csv",
    ISIC_ROOT / "ISIC_2019_Test_Metadata.csv",
    "--output-dir",
    ARTIFACTS_ROOT / "overlap" / "ham10000-isic2019",
)

if EXPORT_PAPER_TABLES:
    run_py(
        "export_ablation_table.py",
        "--runs-root",
        ARTIFACTS_ROOT / "final_evidence",
        "--run-spec",
        "q1-ham10000-convnext-repair-final-e12-multiseed::Repair",
        "--run-spec",
        "q1-ham10000-convnext-no-repair-final-e12-multiseed::No repair",
        "--run-spec",
        "q1-ham10000-convnext-random-repair-final-e12-multiseed::Random repair",
        "--output-dir",
        TABLES_ROOT / "internal_main",
    )
    run_py(
        "export_ablation_table.py",
        "--runs-root",
        ARTIFACTS_ROOT / "final_evidence",
        "--run-spec",
        "q1-ham10000-convnext-repair-final-e12-multiseed::Repair",
        "--run-spec",
        "q1-ham10000-convnext-clean-upper-final-e12-multiseed::Clean upper",
        "--run-spec",
        "q1-ham10000-convnext-gce-no-repair-final-e12-multiseed::GCE no repair",
        "--output-dir",
        TABLES_ROOT / "noisy_anchor",
    )
    run_py(
        "export_ablation_table.py",
        "--runs-root",
        ARTIFACTS_ROOT / "final_evidence",
        "--run-spec",
        "q1-ham10000-convnext-repair-final-e12-multiseed-external-isic2019-test::Repair external",
        "--run-spec",
        "q1-ham10000-convnext-no-repair-final-e12-multiseed-external-isic2019-test::No repair external",
        "--run-spec",
        "q1-ham10000-convnext-random-repair-final-e12-multiseed-external-isic2019-test::Random repair external",
        "--output-dir",
        TABLES_ROOT / "external_main",
    )
else:
    print("Skipped: paper table exports")

guarded_run(
    EXPORT_RESULTS_BUNDLE,
    "Q1 paper bundle export",
    run_py,
    "export_results_bundle.py",
    "--runs-root",
    ARTIFACTS_ROOT / "final_evidence",
    "--run",
    "q1-ham10000-convnext-repair-final-e12-multiseed",
    "--run",
    "q1-ham10000-convnext-no-repair-final-e12-multiseed",
    "--run",
    "q1-ham10000-convnext-random-repair-final-e12-multiseed",
    "--run",
    "q1-ham10000-convnext-clean-upper-final-e12-multiseed",
    "--run",
    "q1-ham10000-convnext-gce-no-repair-final-e12-multiseed",
    "--run",
    "q1-ham10000-convnext-repair-final-e12-multiseed-external-isic2019-test",
    "--run",
    "q1-ham10000-convnext-no-repair-final-e12-multiseed-external-isic2019-test",
    "--run",
    "q1-ham10000-convnext-random-repair-final-e12-multiseed-external-isic2019-test",
    "--output-root",
    EXPORTS_ROOT,
    "--bundle-name",
    "trustquerynet-q1-paper-bundle",
)
guarded_run(
    EXPORT_PAPER_FIGURES,
    "Paper figure export",
    run_py,
    "export_paper_figures.py",
    "--runs-root",
    ARTIFACTS_ROOT / "final_evidence",
    "--repair-run",
    "q1-ham10000-convnext-repair-final-e12-multiseed",
    "--no-repair-run",
    "q1-ham10000-convnext-no-repair-final-e12-multiseed",
    "--random-repair-run",
    "q1-ham10000-convnext-random-repair-final-e12-multiseed",
    "--repair-external-run",
    "q1-ham10000-convnext-repair-final-e12-multiseed-external-isic2019-test",
    "--no-repair-external-run",
    "q1-ham10000-convnext-no-repair-final-e12-multiseed-external-isic2019-test",
    "--random-repair-external-run",
    "q1-ham10000-convnext-random-repair-final-e12-multiseed-external-isic2019-test",
    "--overlap-report",
    ARTIFACTS_ROOT / "overlap" / "ham10000-isic2019" / "overlap_report.json",
    "--output-dir",
    ARTIFACTS_ROOT / "paper_figures",
)

for path in [
    TABLES_ROOT / "internal_main" / "ablation_table.md",
    TABLES_ROOT / "noisy_anchor" / "ablation_table.md",
    TABLES_ROOT / "external_main" / "ablation_table.md",
]:
    show_text(path)


## Results Review

These cells are only for reading generated artifacts after the runs finish. They keep the notebook presentation clean while still giving you obvious places to inspect the final outputs.


In [1]:
baseline_result_paths = [
    ARTIFACTS_ROOT / "runs" / "full-ham10000-convnext-balanced-multiseed" / "aggregate_results.md",
    ABLATIONS_ROOT / "ham10000" / "ablation_table.md",
]

for path in baseline_result_paths:
    show_text(path)



===== /content/drive/MyDrive/TrustQueryNet/artifacts/runs/full-ham10000-convnext-balanced-multiseed/aggregate_results.md =====

| Metric | Mean | Std | Min | Max |
| --- | ---: | ---: | ---: | ---: |
| `best_val_macro_f1` | 0.6995 | 0.0152 | 0.6797 | 0.7164 |
| `test_cal_accuracy` | 0.8074 | 0.0095 | 0.7960 | 0.8182 |
| `test_cal_macro_f1` | 0.6933 | 0.0244 | 0.6676 | 0.7295 |
| `test_cal_ece` | 0.0327 | 0.0053 | 0.0277 | 0.0415 |
| `test_cal_macro_auroc` | 0.9410 | 0.0096 | 0.9292 | 0.9531 |
| `test_cal_coverage_at_0_5` | 0.9313 | 0.0095 | 0.9152 | 0.9401 |
| `test_cal_risk_at_0_5` | 0.1630 | 0.0101 | 0.1522 | 0.1726 |

===== /content/drive/MyDrive/TrustQueryNet/ablations/ham10000/ablation_table.md =====

| label | n_seeds | test_cal_accuracy | test_cal_macro_f1 | test_cal_macro_auroc | test_uncal_ece | test_cal_ece | ece_improvement | test_cal_coverage_at_0_5 | test_cal_risk_at_0_5 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Full model | 5 | 0.8074 ± 0.0095 | 0

In [2]:
internal_final_result_paths = [
    ARTIFACTS_ROOT / "final_evidence" / "q1-ham10000-convnext-repair-final-e12-multiseed" / "aggregate_results.md",
    ARTIFACTS_ROOT / "final_evidence" / "q1-ham10000-convnext-no-repair-final-e12-multiseed" / "aggregate_results.md",
    ARTIFACTS_ROOT / "final_evidence" / "q1-ham10000-convnext-random-repair-final-e12-multiseed" / "aggregate_results.md",
    ARTIFACTS_ROOT / "final_evidence" / "q1-ham10000-convnext-clean-upper-final-e12-multiseed" / "aggregate_results.md",
    ARTIFACTS_ROOT / "final_evidence" / "q1-ham10000-convnext-gce-no-repair-final-e12-multiseed" / "aggregate_results.md",
]

for path in internal_final_result_paths:
    show_text(path)



===== /content/drive/MyDrive/TrustQueryNet/artifacts/final_evidence/q1-ham10000-convnext-repair-final-e12-multiseed/aggregate_results.md =====

| Metric | Mean | Std | Min | Max |
| --- | ---: | ---: | ---: | ---: |
| `best_val_macro_f1` | 0.7250 | 0.0190 | 0.6979 | 0.7508 |
| `test_cal_accuracy` | 0.8350 | 0.0059 | 0.8303 | 0.8424 |
| `test_cal_macro_f1` | 0.7152 | 0.0216 | 0.6916 | 0.7485 |
| `test_cal_ece` | 0.0445 | 0.0090 | 0.0329 | 0.0561 |
| `test_cal_macro_auroc` | 0.9382 | 0.0133 | 0.9234 | 0.9531 |
| `test_cal_aurc` | 0.0728 | 0.0186 | 0.0501 | 0.0944 |
| `test_cal_coverage_at_0_5` | 0.9525 | 0.0074 | 0.9421 | 0.9616 |
| `test_cal_risk_at_0_5` | 0.1428 | 0.0054 | 0.1369 | 0.1499 |

===== /content/drive/MyDrive/TrustQueryNet/artifacts/final_evidence/q1-ham10000-convnext-no-repair-final-e12-multiseed/aggregate_results.md =====

| Metric | Mean | Std | Min | Max |
| --- | ---: | ---: | ---: | ---: |
| `best_val_macro_f1` | 0.7195 | 0.0115 | 0.7071 | 0.7347 |
| `test_cal_accurac

In [3]:
external_final_result_paths = [
    ARTIFACTS_ROOT / "final_evidence" / "q1-ham10000-convnext-repair-final-e12-multiseed-external-isic2019-test" / "aggregate_results.md",
    ARTIFACTS_ROOT / "final_evidence" / "q1-ham10000-convnext-no-repair-final-e12-multiseed-external-isic2019-test" / "aggregate_results.md",
    ARTIFACTS_ROOT / "final_evidence" / "q1-ham10000-convnext-random-repair-final-e12-multiseed-external-isic2019-test" / "aggregate_results.md",
    ARTIFACTS_ROOT / "overlap" / "ham10000-isic2019" / "overlap_report.json",
]

for path in external_final_result_paths:
    show_text(path)



===== /content/drive/MyDrive/TrustQueryNet/artifacts/final_evidence/q1-ham10000-convnext-repair-final-e12-multiseed-external-isic2019-test/aggregate_results.md =====

| Metric | Mean | Std | Min | Max |
| --- | ---: | ---: | ---: | ---: |
| `test_cal_accuracy` | 0.5692 | 0.0145 | 0.5497 | 0.5892 |
| `test_cal_macro_f1` | 0.4427 | 0.0117 | 0.4304 | 0.4570 |
| `test_cal_ece` | 0.2000 | 0.0253 | 0.1607 | 0.2247 |
| `test_cal_macro_auroc` | 0.8125 | 0.0180 | 0.7987 | 0.8382 |
| `test_cal_aurc` | 0.2804 | 0.0299 | 0.2424 | 0.3102 |
| `test_cal_coverage_at_0_5` | 0.8526 | 0.0230 | 0.8168 | 0.8685 |
| `test_cal_risk_at_0_5` | 0.3805 | 0.0228 | 0.3451 | 0.4059 |

===== /content/drive/MyDrive/TrustQueryNet/artifacts/final_evidence/q1-ham10000-convnext-no-repair-final-e12-multiseed-external-isic2019-test/aggregate_results.md =====

| Metric | Mean | Std | Min | Max |
| --- | ---: | ---: | ---: | ---: |
| `test_cal_accuracy` | 0.5630 | 0.0078 | 0.5493 | 0.5687 |
| `test_cal_macro_f1` | 0.4288 | 

In [4]:
paper_table_paths = [
    TABLES_ROOT / "internal_main" / "ablation_table.md",
    TABLES_ROOT / "noisy_anchor" / "ablation_table.md",
    TABLES_ROOT / "external_main" / "ablation_table.md",
]

for path in paper_table_paths:
    show_text(path)

figure_dir = ARTIFACTS_ROOT / "paper_figures"
if figure_dir.exists():
    print("\nGenerated paper figures:")
    for path in sorted(figure_dir.glob("*")):
        print("-", path.name)
else:
    print(f"No paper figures directory found yet: {figure_dir}")



===== /content/drive/MyDrive/TrustQueryNet/artifacts/paper_tables/internal_main/ablation_table.md =====

| label | n_seeds | test_cal_accuracy | test_cal_macro_f1 | test_cal_macro_auroc | test_cal_aurc | test_uncal_ece | test_cal_ece | ece_improvement | test_cal_coverage_at_0_5 | test_cal_risk_at_0_5 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Repair | 5 | 0.8350 ± 0.0059 | 0.7152 ± 0.0216 | 0.9382 ± 0.0133 | 0.0728 ± 0.0186 | 0.0433 ± 0.0077 | 0.0445 ± 0.0090 | -0.0012 | 0.9525 ± 0.0074 | 0.1428 ± 0.0054 |
| No repair | 5 | 0.8321 ± 0.0071 | 0.7145 ± 0.0130 | 0.9460 ± 0.0128 | 0.0622 ± 0.0093 | 0.0428 ± 0.0077 | 0.0356 ± 0.0075 | 0.0072 | 0.9480 ± 0.0066 | 0.1449 ± 0.0078 |
| Random repair | 5 | 0.8319 ± 0.0051 | 0.7105 ± 0.0239 | 0.9521 ± 0.0128 | 0.0588 ± 0.0145 | 0.0420 ± 0.0093 | 0.0356 ± 0.0071 | 0.0064 | 0.9376 ± 0.0100 | 0.1402 ± 0.0107 |

===== /content/drive/MyDrive/TrustQueryNet/artifacts/paper_tables/noisy_anchor/ablation_table.md =====

| label

## Optional Packaging And Browser Downloads

Use these toggles only after the relevant exports exist. The final bundle mirrors the compact artifact package that was previously assembled manually in ad-hoc notebook cells.


In [ ]:
PACKAGE_FIGURES_ZIP = False
PACKAGE_FINAL_HARDENING_BUNDLE = False
DOWNLOAD_ZIPS = False

figure_zip_path = guarded_run(PACKAGE_FIGURES_ZIP, "Package paper figures zip", package_figure_bundle)
final_bundle_result = guarded_run(PACKAGE_FINAL_HARDENING_BUNDLE, "Package final hardening bundle", package_final_bundle)

if DOWNLOAD_ZIPS:
    from google.colab import files

    if figure_zip_path:
        files.download(str(figure_zip_path))
    if final_bundle_result:
        _, final_bundle_zip = final_bundle_result
        files.download(str(final_bundle_zip))
else:
    print("Skipped: browser downloads")
